# Human-in-the-loop correction notebook

**Goal**: review a real-plan inference, mark false positives as DELETE and add missed columns as ADD, and write the corrections to `data/corrections.db`. Run `scripts/retrain_yolo.py` afterwards to fold the corrections into a fine-tune.

Closed feedback loop:
```
review (this notebook) → data/corrections.db → scripts/retrain_yolo.py
  → column_detect_ft_{ts}.pt → manual `cp` to column_detect.pt → next review
```

Schema contract (kept compatible with `scripts/retrain_yolo.py`):
```
data/corrections.db                     — SQLite log of edits/deletes/adds
data/jobs/{job_id}/render.jpg           — the plan image as reviewed
data/jobs/{job_id}/px_detections.json   — { "columns": [{"bbox": [...], ...}, ...] }
```

In [1]:
# Cell 1 — imports
import io
import math
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, clear_output

Image.MAX_IMAGE_PIXELS = None

# Walk up from CWD until we find scripts/corrections_logger.py. Lets the
# notebook be opened from any directory (JupyterLab launched at $HOME,
# VS Code, etc.) without ModuleNotFoundError.
_search = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in _search if (p / 'scripts' / 'corrections_logger.py').exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        'Could not locate scripts/corrections_logger.py walking up from '
        f'{Path.cwd()}. Open this notebook from the project root, or set '
        'ROOT manually below.'
    )
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from corrections_logger import (
    new_job_id, save_job, record_delete, record_add, record_edit, summary,
    JobAlreadyCorrected,
)
from postprocess_pipeline import run_pipeline, DEFAULT_CONFIG, format_audit
from ood_detector import OutOfDistributionError

print(f'project root : {ROOT}')

project root : /home/jiezhi/Documents/1-column-train


In [2]:
# Cell 2 — config
WEIGHTS    = ROOT / 'column_detect.pt'
# Set DRAWING_ID to the id you used with `hitl.py ingest`. The raster
# path is resolved canonically from data/raw/drawings/<id>.meta.json
# in cell 3 — DO NOT hard-code IMAGE_PATH here. Hard-coding bypasses
# the ingest pipeline and corrections will be anchored to a different
# coordinate frame than the model trains on.
DRAWING_ID = 'TGCH-TD-S-200-L3-00'

TILE_SIZE  = 1280
TILE_STEP  = 1080            # 200 px overlap, same as training
CONF_TH    = 0.25            # one of two public inference knobs
INPUT_DPI  = 300             # the other; OOD check rejects far-off values
IOU_TH     = 0.45
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'

print(f'weights    : {WEIGHTS}')
print(f'drawing_id : {DRAWING_ID}')
print(f'device     : {DEVICE}')
print(f'conf_th    : {CONF_TH}')
print(f'input_dpi  : {INPUT_DPI}')

weights    : /home/jiezhi/Documents/1-column-train/column_detect.pt
drawing_id : TGCH-TD-S-200-L3-00
device     : 0
conf_th    : 0.25
input_dpi  : 300


In [3]:
# Cell 3 — resolve canonical raster, load + tiled inference
from tiled_inference import tiled_predict
from ingest_drawings import resolve_drawing

IMAGE_PATH, _meta = resolve_drawing(DRAWING_ID)
print(f'canonical raster: {IMAGE_PATH}')
print(f'ingested at      : {_meta["ingested_ts"]:.0f}  DPI: {_meta["dpi"]}  '
      f'size: {_meta["size"][0]}x{_meta["size"][1]}')

model = YOLO(str(WEIGHTS))
img   = Image.open(IMAGE_PATH).convert('RGB')
W, H  = img.size

all_boxes, all_scores, tile_counts = tiled_predict(
    model, img,
    tile=TILE_SIZE, step=TILE_STEP,
    conf=CONF_TH, iou=IOU_TH,
    device=DEVICE,
)
print(f'tiles processed: {len(tile_counts)}')
print(f'raw detections (pre-NMS): {len(all_boxes)}')
print(f'per-tile dets: min={min(tile_counts)} max={max(tile_counts)} '
      f'mean={sum(tile_counts)/max(1,len(tile_counts)):.2f}')

canonical raster: /home/jiezhi/Documents/1-column-train/data/raw/drawings/TGCH-TD-S-200-L3-00.jpg
ingested at      : 1780471952  DPI: 96  size: 13480x9536
tiles processed: 117
raw detections (pre-NMS): 1162
per-tile dets: min=0 max=73 mean=9.93


## Cell 4 — 7-filter post-processing (shared module)

The pipeline lives in `scripts/postprocess_pipeline.py` so this notebook and `test_column.ipynb` cannot drift. Filter order: aspect → size → shape → OCR-text → centre-NMS → IoU-NMS. OOD hard-fail runs first when `INPUT_DPI` is passed.

Override any constant via a custom `PostprocessConfig(...)` instead of editing the cell.

In [4]:
# Cell 4 — call the shared post-processing pipeline with real tile counts
img_gray = np.asarray(img.convert('L'))

try:
    boxes_final, scores_final, audit = run_pipeline(
        img_gray, all_boxes, all_scores,
        config=DEFAULT_CONFIG,
        input_dpi=INPUT_DPI,
        tile_detection_counts=tile_counts,
    )
except OutOfDistributionError as e:
    print(f'OOD HARD FAILURE: {e}')
    print('Refusing to emit predictions. Fix the rasterisation and re-run.')
    boxes_final  = np.zeros((0, 4), dtype=np.float32)
    scores_final = np.zeros((0,),    dtype=np.float32)
    raise   # surface the failure; remove the raise to demote to a warning

print(format_audit(audit))

raw                   : 1162
after aspect          : 1111
after size            : 1093
after shape           : 981
after OCR text        : 567
after centre-NMS      : 379
FINAL                 : 379


In [5]:
# Cell 5 — register the job (writes data/jobs/{job_id}/render.jpg + px_detections.json)
job_id = new_job_id()
job_dir = save_job(job_id, img, boxes_final, scores_final,
                   source_path=str(IMAGE_PATH))
print(f'job_id   : {job_id}')
print(f'job dir  : {job_dir}')
print(f'detections registered: {len(boxes_final)}')

job_id   : 337f52d41053474fb2f4c631ed1c792c
job dir  : data/jobs/337f52d41053474fb2f4c631ed1c792c
detections registered: 379


## Cell 6 — review grid

Page through detections in batches of 20. Set the dropdown under each thumbnail to **delete** if the bbox is a false positive (text, wall corner, geometry artefact, etc.). Leave **keep** for real columns. The red square in each crop is the detected bbox; the surrounding context shows what's actually under it.

Click **Save corrections** when done. Per-click writes were rejected on purpose (lets you change your mind, no DB spam).

In [6]:
# Cell 6 — review grid as a self-contained class
# Re-executing this cell yields a fresh ReviewGrid; stale grids cannot
# write into a different job_id because each instance binds to the
# job_id passed at construction time.

class ReviewGrid:
    PAGE_SIZE  = 20
    GRID_COLS  = 5
    CROP_PAD   = 14
    THUMB_SIZE = 96

    def __init__(self, job_id, img, boxes, scores):
        self.job_id  = job_id
        self.img     = img
        self.W, self.H = img.size
        self.boxes   = np.asarray(boxes)
        self.scores  = np.asarray(scores)
        self.marks   = ['keep'] * len(self.boxes)
        self.idx     = 0
        self._thumb_cache = [self._crop(b) for b in self.boxes]
        self._grid_out = widgets.Output()
        self._status   = widgets.Label('')

    def _crop(self, b):
        x1, y1, x2, y2 = b
        cx1 = max(0, int(x1 - self.CROP_PAD)); cy1 = max(0, int(y1 - self.CROP_PAD))
        cx2 = min(self.W, int(x2 + self.CROP_PAD))
        cy2 = min(self.H, int(y2 + self.CROP_PAD))
        crop = self.img.crop((cx1, cy1, cx2, cy2)).copy()
        d = ImageDraw.Draw(crop)
        d.rectangle([(x1 - cx1, y1 - cy1), (x2 - cx1, y2 - cy1)],
                    outline=(220, 30, 30), width=2)
        crop = crop.resize((self.THUMB_SIZE, self.THUMB_SIZE))
        buf = io.BytesIO()
        crop.save(buf, format='PNG')
        return buf.getvalue()

    def _make_tile(self, i):
        img_w = widgets.Image(value=self._thumb_cache[i], format='png',
                              width=self.THUMB_SIZE, height=self.THUMB_SIZE)
        dd = widgets.Dropdown(
            options=[('keep', 'keep'), ('DELETE', 'delete')],
            value=self.marks[i],
            layout=widgets.Layout(width=f'{self.THUMB_SIZE}px'),
        )
        def on_change(change, idx=i, self=self):
            self.marks[idx] = change['new']
        dd.observe(on_change, names='value')
        label = widgets.Label(f'#{i}  c={float(self.scores[i]):.2f}',
                              layout=widgets.Layout(width=f'{self.THUMB_SIZE}px'))
        return widgets.VBox([img_w, dd, label],
                            layout=widgets.Layout(margin='4px',
                                                   border='1px solid #ddd',
                                                   padding='4px'))

    def _render(self):
        n = len(self.boxes)
        pages = max(1, (n + self.PAGE_SIZE - 1) // self.PAGE_SIZE)
        start = self.idx * self.PAGE_SIZE
        end   = min(start + self.PAGE_SIZE, n)
        tiles = [self._make_tile(i) for i in range(start, end)]
        rows = [widgets.HBox(tiles[r:r+self.GRID_COLS])
                for r in range(0, len(tiles), self.GRID_COLS)]
        with self._grid_out:
            clear_output(wait=True)
            display(widgets.VBox(rows))
            n_del = sum(1 for m in self.marks if m == 'delete')
            print(f'Page {self.idx+1}/{pages}    '
                  f'showing {start}-{end-1 if n else 0} of {n}    '
                  f'marked DELETE so far: {n_del}')

    def _prev(self, _):
        if self.idx > 0:
            self.idx -= 1; self._render()

    def _next(self, _):
        n = len(self.boxes)
        pages = max(1, (n + self.PAGE_SIZE - 1) // self.PAGE_SIZE)
        if self.idx < pages - 1:
            self.idx += 1; self._render()

    def _save(self, _):
        n_deleted = 0
        for i, m in enumerate(self.marks):
            if m == 'delete':
                try:
                    record_delete(self.job_id, i)
                    n_deleted += 1
                except Exception as e:
                    print(f'  ! delete {i} failed: {e}')
        s = summary()
        self._status.value = (
            f'Saved {n_deleted} delete corrections. '
            f'DB: {s["deletes"]} deletes / {s["edits_or_adds"]} edits-or-adds '
            f'across {s["jobs"]} job(s).'
        )
        print(self._status.value)

    def display(self):
        prev_btn = widgets.Button(description='← Prev')
        next_btn = widgets.Button(description='Next →')
        save_btn = widgets.Button(description='Save corrections',
                                  button_style='primary')
        prev_btn.on_click(self._prev)
        next_btn.on_click(self._next)
        save_btn.on_click(self._save)
        display(widgets.HBox([prev_btn, next_btn, save_btn, self._status]))
        display(self._grid_out)
        self._render()


grid = ReviewGrid(job_id, img, boxes_final, scores_final)
grid.display()

Output()

## Cell 7 — ADD missed columns (FN corrections)

For columns the model **missed entirely** (no thumbnail above to mark), open `L3.jpg` in your image viewer, hover each missed column, read off (cx, cy) and a rough size_px (e.g., 24 for a small filled square, 32 for a typical outlined one). Add tuples to the `missed` list below, then run the cell. Each entry becomes a row in `data/corrections.db` with `is_delete=0` and a `bbox` derived from (cx, cy, size_px).

You can iterate: run this cell, check `summary()`, add more entries, run again. Each call appends; nothing is overwritten.

In [8]:
# Cell 7 — ADD missed columns
# Format: (cx, cy, size_px). The bbox becomes (cx - sz/2, cy - sz/2, cx + sz/2, cy + sz/2).
# Re-running this cell with the same list does NOT double-append; record_add
# dedups by (job_id, round(cx), round(cy)).
missed = [
    # (1234, 5678, 28),
]

n_added = 0
n_skipped = 0
for cx, cy, sz in missed:
    bbox = [cx - sz/2.0, cy - sz/2.0, cx + sz/2.0, cy + sz/2.0]
    if record_add(job_id, bbox):
        n_added += 1
    else:
        n_skipped += 1
print(f'Added {n_added} missed-column corrections ({n_skipped} duplicates skipped).')
print('Corrections DB summary:', summary())

Added 0 missed-column corrections (0 duplicates skipped).
Corrections DB summary: {'jobs': 1, 'corrections': 172, 'deletes': 172, 'edits_or_adds': 0, 'rescinded_deletes': 0}


## Cell 8 — retrain handoff

When `scripts/retrain_yolo.py` sees enough corrections (default ≥10, configurable via `--min-corrections`), it folds them into a YOLO fine-tune from `column_detect.pt` and writes a timestamped weight to the project root. Promote it manually after inspecting on a real plan.

In [9]:
# Cell 8 — print the retrain command
s = summary()
print(f'Current DB: {s["jobs"]} job(s), {s["corrections"]} corrections '
      f'({s["deletes"]} deletes, {s["edits_or_adds"]} edits/adds)')
print()
print('To fold corrections into a fine-tune, run from project root:')
print('  python3 scripts/retrain_yolo.py --epochs 30 --min-corrections 10')
print()
print('Dry-run (build dataset, skip training):')
print('  python3 scripts/retrain_yolo.py --dry-run')
print()
print('Output weight will be column_detect_ft_{timestamp}.pt at project root.')
print('Inspect on a real plan, then manually promote:')
print('  cp column_detect_ft_<ts>.pt column_detect.pt')

Current DB: 1 job(s), 172 corrections (172 deletes, 0 edits/adds)

To fold corrections into a fine-tune, run from project root:
  python3 scripts/retrain_yolo.py --epochs 30 --min-corrections 10

Dry-run (build dataset, skip training):
  python3 scripts/retrain_yolo.py --dry-run

Output weight will be column_detect_ft_{timestamp}.pt at project root.
Inspect on a real plan, then manually promote:
  cp column_detect_ft_<ts>.pt column_detect.pt
